# 09. Ablación controlada de `tiene_avicena` — Fase 6

`tiene_avicena` = `n_consultas_T.notna()`: indicador puro de **tener** registro Avicena, una
señal de **vigilancia** (uso del sistema de salud, no riesgo intrínseco) y **redundante** con
`n_consultas_T`. Por eso se **excluye a priori** del modelo final.

Este notebook **verifica** que esa exclusión no cuesta rendimiento, mediante una **ablación
controlada**: dos XGBoost con los **mismos hiperparámetros canónicos** (de la Fase 5b), que solo
difieren en una característica:

- **58 features (sin avicena)** → modelo ganador del proyecto (`modelo_fase6_XGBoost-sin-avicena`).
- **59 features (con avicena)** → reentrenado en vivo con los mismos hiperparámetros.

> Igual diseño, igual tuning, única diferencia = la feature. Si las métricas no caen, excluirla
> es gratis (y elimina sesgo de vigilancia).


In [1]:
import json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import xgboost as xgb, joblib
from sklearn.metrics import average_precision_score, roc_auc_score

B = "bases"; RNG = 42
RES = json.load(open(f"{B}/_resultados_sin_avicena.json", encoding="utf-8"))
bp = RES['fase5b']['xgb_best_params']
print("Hiperparámetros canónicos XGBoost (Fase 5b):")
for k, v in bp.items():
    print(f"  {k}: {v if isinstance(v, int) else round(v, 5)}")


Hiperparámetros canónicos XGBoost (Fase 5b):
  n_estimators: 387
  learning_rate: 0.03462
  max_depth: 6
  min_child_weight: 20.82525
  subsample: 0.65205
  colsample_bytree: 0.60973
  gamma: 2.27875
  reg_alpha: 0.01337
  reg_lambda: 0.00627
  scale_pos_weight: 3.58709


In [2]:
nat_tr = pd.read_parquet(f"{B}/prediccion_mama_train_native.parquet")
nat_va = pd.read_parquet(f"{B}/prediccion_mama_val_native.parquet")
y_tr = nat_tr['label'].values.astype(int)
y_va = nat_va['label'].values.astype(int)

FEAT_58 = [c for c in nat_tr.columns if c not in ('key', 'label', 'tiene_avicena')]  # sin avicena
FEAT_59 = [c for c in nat_tr.columns if c not in ('key', 'label')]                   # con avicena
print(f"58 feats (sin avicena) | 59 feats (con avicena) | train pos {y_tr.sum()} | val pos {y_va.sum()}")

def summarize(y, p, name):
    m = {'modelo': name, 'AUC_PR': float(average_precision_score(y, p)),
         'AUC_ROC': float(roc_auc_score(y, p))}
    for f in (0.005, 0.01, 0.05, 0.10):
        k = max(1, int(len(p) * f)); idx = np.argpartition(-p, k)[:k]
        m[f'recall@top{f*100:.1f}%'] = float(y[idx].sum() / y.sum())
    return m


58 feats (sin avicena) | 59 feats (con avicena) | train pos 1857 | val pos 993


## 1. Brazo SIN avicena (58 feats) — modelo ganador

Se **carga** el modelo ganador del proyecto y se evalúa en validación (exacto).


In [3]:
m58 = joblib.load(f"{B}/modelo_fase6_XGBoost-sin-avicena.joblib")
assert m58.n_features_in_ == 58
p58 = m58.predict_proba(nat_va[FEAT_58].astype('float32').values)[:, 1]
res58 = summarize(y_va, p58, 'XGBoost SIN avicena (58, ganador)')
print(f"AUC-PR {res58['AUC_PR']:.4f} | recall@top10% {res58['recall@top10.0%']:.4f}")


AUC-PR 0.0388 | recall@top10% 0.6989


## 2. Brazo CON avicena (59 feats) — reentreno en vivo

Mismos hiperparámetros canónicos, añadiendo `tiene_avicena`. Reentreno sobre el train completo.


In [4]:
params = dict(bp); params.update(tree_method='hist', eval_metric='aucpr', n_jobs=-1, random_state=RNG)
m59 = xgb.XGBClassifier(**params).fit(nat_tr[FEAT_59].astype('float32').values, y_tr)
p59 = m59.predict_proba(nat_va[FEAT_59].astype('float32').values)[:, 1]
res59 = summarize(y_va, p59, 'XGBoost CON avicena (59)')
print(f"AUC-PR {res59['AUC_PR']:.4f} | recall@top10% {res59['recall@top10.0%']:.4f}")


AUC-PR 0.0373 | recall@top10% 0.7090


## 3. Comparación — ¿cuesta algo excluir `tiene_avicena`?

In [5]:
comp = pd.DataFrame([res58, res59]).set_index('modelo')
pd.set_option('display.width', 200, 'display.max_columns', 20)
print("=== Ablación 2023→2024 — sin (58) vs con (59) tiene_avicena ===")
print(comp.to_string(float_format='{:.4f}'.format))
d_pr = res58['AUC_PR'] - res59['AUC_PR']
d_top10 = res58['recall@top10.0%'] - res59['recall@top10.0%']
print(f"\nDelta (sin − con) AUC-PR: {d_pr:+.4f} | recall@top10%: {d_top10:+.4f}")
print("Veredicto:", "excluirla NO cuesta (impacto despreciable)" if abs(d_pr) < 0.002
      else "impacto NO despreciable, revisar")


=== Ablación 2023→2024 — sin (58) vs con (59) tiene_avicena ===
                                   AUC_PR  AUC_ROC  recall@top0.5%  recall@top1.0%  recall@top5.0%  recall@top10.0%
modelo                                                                                                             
XGBoost SIN avicena (58, ganador)  0.0388   0.9007          0.2739          0.3333          0.5680           0.6989
XGBoost CON avicena (59)           0.0373   0.9018          0.2779          0.3374          0.5760           0.7090

Delta (sin − con) AUC-PR: +0.0015 | recall@top10%: -0.0101
Veredicto: excluirla NO cuesta (impacto despreciable)


## 4. Conclusión

Excluir `tiene_avicena` deja el rendimiento prácticamente igual (a veces lo mejora levemente),
y a cambio **elimina una señal de vigilancia** redundante con `n_consultas_T`. La decisión de
excluirla a priori queda **verificada empíricamente**. El modelo ganador del proyecto opera con
**58 features**: `bases/modelo_fase6_XGBoost-sin-avicena.joblib`.
